In [ ]:
import sys
from pathlib import Path

_root = next(p for p in Path.cwd().resolve().parents if (p / '.git').exists() or (p / 'setup.py').exists())
for _p in (str(_root), str(_root / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import scanpy as sc
import matplotlib.pyplot as plt

from metab_processing.metab_travlr_config import DATA_DIR
from metab_processing.SpaceTravLR.dataset_configs import dataset_paths
from metab_processing.LinearRegression.least_squares import (
    fit_gene_betas, rank_coefficients, plot_top_coefficients,
    subsample_metab_betas, plot_beta_histogram)

In [ ]:
data_dir = f'{DATA_DIR}/Alexi_UC_Spliced'
samples = [f'13473_HS4_UC-Slice_{i}' for i in (1, 2, 3, 4)]

ANNOT_COL = '25_06_11_ICI_5K_Coarse_annotations'
T_CELL   = 'T'          # <- set to the T-cell label in ANNOT_COL (see inspect cell below)
GLUCOSE  = 'D-Glucose'  # <- set to glucose's name in uns['x_metab_modulators']
HIST_GENE = None        # None -> first stored gene; or set a specific target gene
N_SUB, FRAC = 200, 0.8

def load_x(dataset):
    return sc.read_h5ad(dataset_paths(dataset, data_dir=data_dir)['dataset_dir'] / 'LinearRegression' / 'x_adata.h5ad')

In [ ]:
# Inspect one sample to set T_CELL / GLUCOSE above if the defaults don't match.
ad0 = load_x(samples[0])
print('annotations:', list(ad0.obs[ANNOT_COL].unique()))
print('metabolites:', [m.split('@', 1)[1] for m in ad0.uns['x_metab_modulators']])
print('genes:', ad0.uns['x_genes'])

In [ ]:
# T cells in the relevant sample: OLS betas ranked by |magnitude|.
betas = fit_gene_betas(ad0, annot_col=ANNOT_COL, annot_value=T_CELL)
display(rank_coefficients(betas).head(20))
plot_top_coefficients(betas, top=20).set_title(f'{samples[0]} - T cells');

In [ ]:
# Run on all samples: top T-cell coefficients per sample.
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for dataset, ax in zip(samples, axes.ravel()):
    b = fit_gene_betas(load_x(dataset), annot_col=ANNOT_COL, annot_value=T_CELL)
    plot_top_coefficients(b, top=15, ax=ax)
    ax.set_title(f'{dataset} - T cells')
fig.tight_layout()

In [ ]:
# Cell-subsampling test, glucose only: distribution of the glucose beta per sample (4 histograms).
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for dataset, ax in zip(samples, axes.ravel()):
    adx = load_x(dataset)
    gene = HIST_GENE or adx.uns['x_genes'][0]
    vals = subsample_metab_betas(adx, gene, GLUCOSE, n_subsamples=N_SUB, frac=FRAC,
                                 annot_col=ANNOT_COL, annot_value=T_CELL)
    plot_beta_histogram(vals, ax=ax, title=f'{dataset}\n{gene} ~ {GLUCOSE} (T cells)')
    ax.set_xlabel('glucose beta')
fig.tight_layout()